In [ ]:
import pandas as pd
import spacy
import spacy_syllables
import os
from num2words import num2words
import numpy as np

df1 = pd.read_csv("durationibiscolona.csv")

df1["filename"] = df1["filename"].apply(lambda x: os.path.splitext(x)[0])

df1

In [ ]:
df2= pd.read_csv("ibiscolonbelmentions.csv")
df2

In [ ]:
import pandas as pd
df = pd.read_csv("Level_Analysis/combined_word_syll.csv")
df["words/turn"] = df["n_words"]/df["turns"]
df_doctor_agg = (
    df[df["label"] == "Doctor"]
    .groupby("filename", as_index=False)["n_words"]
    .sum()
    .merge(
        df2[df2["label"] == "Doctor"][["filename", "Percentage_BEL_mentions"]],
        on="filename",
        how="inner"
    )
)

In [ ]:
df_doctor_agg = df_doctor_agg[["filename","Percentage_BEL_mentions"]]

In [ ]:
df_doctor_agg

In [ ]:
df3 = pd.merge(df2, df1,on="filename",
    how="inner")

In [ ]:
df3["n_words"] = pd.to_numeric(df3["n_words"], errors="coerce")
df3["total_syllables"] = pd.to_numeric(df3["total_syllables"], errors="coerce")
df3["duration"] = pd.to_numeric(df3["duration"], errors="coerce")

df3 = df3.dropna(subset=["n_words", "total_syllables", "duration"])

# Aggregate per filename
df_grouped = (
    df3.groupby("filename")
      .agg({
          "total_syllables": "sum",
          "n_words": "sum",
          "duration": "first"
      })
      .reset_index()
)

df_grouped["syllables_per_second"] = df_grouped["total_syllables"] / df_grouped["duration"].astype(int)
df_grouped["words_per_second"] = df_grouped["n_words"] / df_grouped["duration"].astype(int)


# Keep only what you need
df_final = df_grouped[["filename", "syllables_per_second", "words_per_second"]]
df_final.to_csv("Level_Analysis/ibis_dialogue.csv")

In [ ]:
df_level = pd.read_csv("Level_Analysis/IBIS_Level_Scores.csv")
df_doc = df_level[df_level["label"] == "Doctor"].copy()

In [ ]:
df_analysis = pd.merge(
    df_final,
    df_doc,
    on="filename",
    how="inner"
)

In [ ]:
df_analysis

In [ ]:
from scipy.stats import spearmanr
import pandas as pd

target = "syllables_per_second"

metrics = [
    "fres"
]
# metrics = [
#     "Percentage_BEL_mentions"
# ]


results = []

for col in metrics:
    r, p = spearmanr(df_analysis[target], df_analysis[col])
    results.append({"metric": col, "spearman_r": r, "p_value": p})

corr_df = pd.DataFrame(results)
print(corr_df)

In [ ]:
import matplotlib.pyplot as plt

df_doc = df[df["label"]=="Doctor"]["words/turn"]
df_patient = df[df["label"]=="Patient"]["words/turn"]

# Plot
plt.figure(figsize=(6, 5))
plt.boxplot([df_doc, df_patient], labels=["Doctor", "Patient"])
plt.show()
